Here is the exact mathematical formulation and implementation strategy for both loss functions. Designing these correctly is what transitions the pipeline from standard computer vision to true physics-aware modeling.

### Phase 1: Enhancement Loss (EDSR)

For the thermal super-resolution step (200m to 100m), you are training on synthetic pairs created by downscaling the native 100m data. The primary objective is pixel accuracy, but the physics constraint is **Spatial Energy Conservation**.

If the model predicts the temperature of four 100m pixels, their average must mathematically equal the original 200m pixel they were derived from.

$$\mathcal{L}_{enhancement} = \mathcal{L}_{L1} + \lambda_{physics} \mathcal{L}_{conservation}$$

* **Standard Reconstruction ($\mathcal{L}_{L1}$):** Compares the predicted 100m patch ($\hat{T}_{100}$) against the downscaled 100m ground truth ($T_{100}$).

$$\mathcal{L}_{L1} = \frac{1}{N} \sum \left| \hat{T}_{100} - T_{100} \right|$$


* **Conservation Penalty ($\mathcal{L}_{conservation}$):** You apply a $2\times2$ Average Pooling operation to the network's high-resolution output and compare it directly to the 200m input ($T_{200}$).

$$\mathcal{L}_{conservation} = \frac{1}{M} \sum \left| \text{AvgPool}(\hat{T}_{100}) - T_{200} \right|$$



**Implementation Note:** In PyTorch, using `torch.nn.functional.avg_pool2d` makes this calculation virtually free in terms of computation, keeping your inference and training times highly optimized for a 6GB VRAM environment. Set $\lambda_{physics}$ relatively high (e.g., $0.5$ to $1.0$) to force the model to respect the thermodynamics early in training.

---

### Phase 2: Colorization Loss (Attention U-Net)

For the Attention U-Net transforming the 100m TIR into 100m RGB, a standard L1 loss will result in blurry, washed-out images. The multi-component loss strategy you outlined is perfectly suited to fix this.

$$\mathcal{L}_{total} = \mathcal{L}_{1} + \lambda_{g} \mathcal{L}_{gradient} + \lambda_{r} \mathcal{L}_{range}$$

Here is how to calculate each term mathematically:

**1. Pixel Reconstruction ($\mathcal{L}_{1}$)**
Supervises accurate global color mapping.


$$\mathcal{L}_{1} = \left\| \hat{I}_{RGB} - I_{RGB} \right\|_1$$

**2. Gradient Loss ($\mathcal{L}_{gradient}$)**
This is the structural enforcer. It ensures that sharp temperature transitions in the thermal map (like a hot road meeting a cool forest) result in equally sharp visual edges in the generated RGB. In simple words, Gradient loss preserves the spatial gradients (edges and intensity transitions) of the enhanced thermal image during colorization, helping maintain image sharpness and fine structural details. You compute this by applying a Sobel filter ($\nabla$) to extract the horizontal and vertical gradients of both images.


$$\mathcal{L}_{gradient} = \left\| \nabla_x \hat{I}_{RGB} - \nabla_x I_{RGB} \right\|_1 + \left\| \nabla_y \hat{I}_{RGB} - \nabla_y I_{RGB} \right\|_1$$

**3. Range Penalty ($\mathcal{L}_{range}$)**
Neural networks can easily output values outside valid physical bounds (e.g., generating negative reflectance values). If your input and target tensors are normalized between $[0, 1]$, this penalty forces the network to stay within that physically plausible range.


$$\mathcal{L}_{range} = \sum \text{ReLU}(\hat{I}_{RGB} - 1.0) + \sum \text{ReLU}(-\hat{I}_{RGB})$$


*By using ReLU, this term evaluates to exactly 0 if the pixel is valid, but aggressively penalizes the network if it overshoots.*

### The Joint-Physics "Secret Weapon"

If you want to push the physics-aware constraint even further without adding overhead to your inference time, you can add a lightweight **Thermo-Biological Penalty** during training.

Healthy vegetation (high NIR reflectance, low Red reflectance) is naturally cooler than built-up urban structures due to evapotranspiration. If the U-Net predicts dense green pixels (high G channel) but the corresponding TIR input patch indicates extreme heat, the network is hallucinating.

You can add a simple tensor multiplication penalty:


$$\mathcal{L}_{ThermoBio} = \text{Mean}(\hat{I}_{Green} \times T_{normalized})$$


This mathematically discourages the generator from putting high green values in high heat areas, solidifying the physical logic of the model.
